# VAN ROI y TIR de dos inversiones

Sistemas de Big Data 5074 · UT1.2 · Diapositivas 12–16.

Versión para completar. Puedes trabajar en este cuaderno en lugar de `alumno.py`. Sustituye los `NotImplementedError` por tu código. Las celdas con tareas pendientes se detendrán hasta que las completes.

Solo biblioteca estándar de Python 3.10 o posterior. Los datos están incluidos. No requiere instalaciones ni archivos externos. Puedes subir este `.ipynb` a Colab. Los ejemplos MapReduce se simulan localmente y en serie, sin ejecutar Hadoop.


# Ejercicio 2. Indicadores económicos de un proyecto

Trabajaremos con las dos tablas de las diapositivas 14 y 15. Cada tabla representa un caso diferente. La diapositiva 13 muestra el logotipo de Colab, pero no aporta un enlace real al cuaderno. El archivo `alumno.py` contiene los datos y el código de apoyo necesarios para empezar.

La simulación usa unidades monetarias porque la presentación no concreta una moneda. Interpretamos «Benefic» como cobros antes de restar los costes de la tabla. La inversión inicial se paga en el momento cero y los demás cobros y pagos ocurren al final de cada año. Solo usamos los importes aportados.

El objetivo es calcular los flujos, el VAN y el ROI e interpretar la TIR. Las validaciones de datos y la bisección para obtener la TIR se proporcionan como apoyo. No tienes que programarlas ni explicar su algoritmo numérico.

## 1. Leer los datos

El caso A tiene una inversión inicial de 33900 y dura cuatro años. El caso B tiene una inversión inicial de 28000 y dura cinco años. En ambos usamos una tasa anual del 3 %, que en Python se escribe `0.03`.

Localiza las constantes `CASO_A`, `CASO_B`, `INVERSION_CASO_A`, `INVERSION_CASO_B` y `TASA_DESCUENTO`. Los CSV de la carpeta `datos` contienen las mismas filas. El programa ya incluye los datos y puede ejecutarse sin cargar esos archivos.

Antes de programar, explica qué representa cada columna. Comprueba también si las dos tablas mantienen los mismos datos en el año 4.

## 2. Calcular los flujos anuales

Para cada año utilizaremos esta relación:

`Ft = beneficio del año − coste del año`

Completa `flujos_anuales(costes, beneficios)` para devolver una lista con un flujo por año. Escribe el bucle y la resta. Conserva las comprobaciones de datos que ya están preparadas.

En el caso B, contrasta el Ft impreso de 11000 en el año 5 con sus columnas de beneficio y coste. Anota qué dato usarías para que la tabla cumpla la relación anterior y explica la discrepancia.

## 3. Añadir la inversión inicial

Completa `construir_flujos(inversion, costes, beneficios)`. El primer elemento será el desembolso inicial con signo negativo. Después irán los flujos de los años 1, 2 y siguientes.

Comprueba que la lista tiene un elemento más que el número de años. Explica por qué un flujo anual también podría ser negativo.

## 4. Calcular el VAN

El VAN expresa todos los flujos en el momento inicial y suma sus valores:

`VAN = CF0 + CF1/(1+k) + CF2/(1+k)**2 + ... + CFn/(1+k)**n`

Completa `van(tasa, flujos)`. Recorre la lista con su número de periodo, calcula el valor actualizado y acumúlalo. El primer periodo es cero. Guarda el resultado sin redondearlo.

Para cada caso, escribe qué significa que el VAN sea positivo, negativo o cero cuando usamos el 3 % como tasa de referencia.

## 5. Calcular el ROI

En este ejercicio usaremos una definición concreta:

`ROI = (suma de los flujos anuales − inversión inicial) / inversión inicial`

Completa `roi(inversion, flujos_anuales)`. El segundo argumento contiene solo los flujos de los años 1 en adelante. No contiene el desembolso inicial. La función devolverá un ratio. Multiplica por 100 al mostrar un porcentaje.

Explica por qué este ROI, por sí solo, no indica cuánto ha durado el proyecto.

## 6. Usar la función TIR

La función `tir(flujos)` ya está preparada. No tienes que escribir el algoritmo de bisección. Úsala con la lista completa que empieza en el momento cero.

La TIR es la tasa que deja el VAN aproximadamente en cero. Compruébalo calculando `van(tir(flujos), flujos)` después de completar tu función VAN. Compara la TIR con el 3 % de referencia. La TIR de un proyecto previsto no garantiza el rendimiento de un depósito bancario.

## 7. Resolver las dos tablas

Completa `resolver_caso` y `resolver_ejercicio`. En `resolver_caso`, llama a `construir_flujos` para formar la serie completa. Usa esa misma serie para calcular VAN y TIR, y excluye el primer elemento cuando necesites solo los flujos anuales. Para cada caso, conserva en un diccionario la inversión, la tasa, los flujos anuales, los flujos desde el momento cero, el VAN, el ROI y la TIR.

Presenta el VAN en unidades monetarias. Muestra ROI y TIR como porcentajes, indicando que el ROI corresponde al conjunto del proyecto y que la TIR es anual bajo estos supuestos.

## 8. Revisar el Ft impreso

Dentro del caso B, prepara una copia de los flujos anuales y cambia únicamente el último por el 11000 impreso. Conserva intactos los datos base. Calcula otra vez los indicadores y etiqueta la salida como sensibilidad, indicando que ese flujo no concuerda con las columnas de la tabla.

Compara la conclusión al 3 % de ambos escenarios. Explica por qué conviene comprobar las restas antes de interpretar los indicadores.


## Código 1 Presentación


In [22]:
"""Ejercicio 2: plantilla del alumnado para las diapositivas 12–16.

Simulación didáctica. Python 3.10+, solo biblioteca estándar.
Los datos están incluidos para ejecutar el archivo o sus celdas en Colab.
"""


'Ejercicio 2: plantilla del alumnado para las diapositivas 12–16.\n\nSimulación didáctica. Python 3.10+, solo biblioteca estándar.\nLos datos están incluidos para ejecutar el archivo o sus celdas en Colab.\n'

## Código 2 Importaciones


In [23]:
from math import fsum, isfinite
from numbers import Real
from pprint import pprint


## Código 3 Datos de las dos tablas


In [24]:
# Unidades monetarias: la presentación no especifica una moneda.
# Interpretamos «Benefic» como cobros antes de restar los costes indicados.
# La inversión inicial es un desembolso separado en t=0.
# Los cobros y pagos de cada año se sitúan al final de ese año.
TASA_DESCUENTO = 0.03
INVERSION_CASO_A = 33900.0
INVERSION_CASO_B = 28000.0

CASO_A = [
    {"anio": 1, "coste": 15500.0, "beneficio": 3000.0},
    {"anio": 2, "coste": 8000.0, "beneficio": 20800.0},
    {"anio": 3, "coste": 10000.0, "beneficio": 26600.0},
    {"anio": 4, "coste": 12000.0, "beneficio": 39800.0},
]

CASO_B = [
    {"anio": 1, "coste": 15500.0, "beneficio": 3000.0},
    {"anio": 2, "coste": 8000.0, "beneficio": 20800.0},
    {"anio": 3, "coste": 10000.0, "beneficio": 26600.0},
    {"anio": 4, "coste": 22000.0, "beneficio": 30000.0},
    {"anio": 5, "coste": 32000.0, "beneficio": 35000.0},
]


## Código 4 Validación de datos

Este bloque es apoyo ya preparado. No hace falta programar la bisección ni memorizar estas validaciones para resolver los indicadores.


In [25]:
def _numero_finito(valor, nombre):
    if isinstance(valor, bool) or not isinstance(valor, Real):
        raise ValueError(f"{nombre} debe ser un número real.")
    try:
        numero = float(valor)
    except OverflowError as error:
        raise ValueError(f"{nombre} queda fuera del rango numérico.") from error
    if not isfinite(numero):
        raise ValueError(f"{nombre} debe ser finito.")
    return numero


def _serie_finita(valores, nombre):
    try:
        serie = [
            _numero_finito(valor, f"{nombre}[{indice}]")
            for indice, valor in enumerate(valores)
        ]
    except TypeError as error:
        raise ValueError(f"{nombre} debe ser una secuencia de números.") from error
    if not serie:
        raise ValueError(f"{nombre} no puede estar vacío.")
    return serie


def _inversion_positiva(inversion):
    inversion = _numero_finito(inversion, "inversion")
    if inversion <= 0:
        raise ValueError("La inversión inicial debe ser mayor que cero.")
    return inversion


def extraer_columnas(datos):
    """Extrae costes y beneficios de años consecutivos 1, 2, ..., n."""
    if not datos:
        raise ValueError("El caso debe tener al menos un año.")
    costes = []
    beneficios = []
    for anio_esperado, fila in enumerate(datos, start=1):
        anio = fila.get("anio")
        if isinstance(anio, bool) or not isinstance(anio, int) or anio != anio_esperado:
            raise ValueError("Los años deben ser enteros consecutivos desde 1.")
        costes.append(fila["coste"])
        beneficios.append(fila["beneficio"])
    return costes, beneficios


## Código 5 Flujos anuales e inversión inicial


In [26]:
def flujos_anuales(costes, beneficios):
    """Devuelve una lista con beneficio - coste para cada año."""
    costes = _serie_finita(costes, "costes")
    beneficios = _serie_finita(beneficios, "beneficios")
    if len(costes) != len(beneficios):
        raise ValueError("Debe haber un coste y un beneficio por año.")
    if any(valor < 0 for valor in costes + beneficios):
        raise ValueError("En este modelo, costes y beneficios son no negativos.")

    # TAREA: Bucle y resta de beneficio - coste por año
    return [b - c for c, b in zip(costes, beneficios)]


def construir_flujos(inversion, costes, beneficios):
    """Añade el desembolso inicial negativo antes de F1, ..., Fn."""
    inversion = _inversion_positiva(inversion)
    f_anuales = flujos_anuales(costes, beneficios)

    # TAREA: El primer elemento es la inversión con signo negativo
    return [-inversion] + f_anuales

## Código 6 Valor actual neto


In [27]:
def van(tasa, flujos):
    """Actualiza y suma los flujos desde el periodo cero."""
    tasa = _numero_finito(tasa, "tasa")
    if tasa <= -1:
        raise ValueError("La tasa debe ser mayor que -1.")
    flujos = _serie_finita(flujos, "flujos")

    # TAREA: Recorrer con el número de período, actualizar y acumular
    valor_actual_neto = 0.0
    for periodo, flujo in enumerate(flujos):
        valor_actual_neto += flujo / ((1.0 + tasa) ** periodo)
    return valor_actual_neto

## Código 7 ROI sobre la inversión inicial


In [28]:
def roi(inversion, flujos_anuales):
    """Devuelve (suma de Ft - I0) / I0 como ratio."""
    inversion = _inversion_positiva(inversion)
    flujos_anuales = _serie_finita(flujos_anuales, "flujos_anuales")

    # TAREA: (suma de flujos anuales - inversión inicial) / inversión inicial
    suma_flujos = fsum(flujos_anuales)
    return (suma_flujos - inversion) / inversion

## Código 8 TIR: función de apoyo numérico ya preparada

Este bloque es apoyo ya preparado. No hace falta programar la bisección ni memorizar estas validaciones para resolver los indicadores.


In [29]:
def tir(flujos):
    """TIR por bisección para flujos convencionales y periodos iguales.

    Exige CF0 < 0 y una única transición de negativo a positivo, ignorando
    ceros. Admite más desembolsos antes del primer cobro neto positivo y
    admite TIR negativa. Busca una raíz r > -1 y devuelve un ratio.

    No es un método general para flujos que alternan varias veces de signo.
    No se pide al alumnado que programe este algoritmo numérico.
    """
    valores = _serie_finita(flujos, "flujos")
    if valores[0] >= 0:
        raise ValueError("La TIR de este modelo requiere un flujo inicial negativo.")
    hay_positivo = False
    for flujo in valores:
        if flujo > 0:
            hay_positivo = True
        elif flujo < 0 and hay_positivo:
            raise ValueError("La TIR admite una sola transición de negativo a positivo.")
    if not hay_positivo:
        raise ValueError("Hace falta al menos un flujo positivo para calcular la TIR.")

    # Los ceros finales no cambian la raíz. La escala evita importes enormes.
    while valores[-1] == 0:
        valores.pop()
    escala = max(abs(valor) for valor in valores)
    valores = [valor / escala for valor in valores]
    ultimo_periodo = len(valores) - 1

    def valor_para_biseccion(tasa):
        base = 1 + tasa
        if base < 1:
            # Es VAN * base**n: conserva su signo para r > -1.
            # En r=-1 solo usamos la extensión del polinomio como frontera.
            return fsum(
                flujo * base ** (ultimo_periodo - periodo)
                for periodo, flujo in enumerate(valores)
            )
        inversa = 1 / base
        return fsum(
            flujo * inversa ** periodo
            for periodo, flujo in enumerate(valores)
        )

    inferior = -1.0
    superior = 1.0
    for _ in range(1024):
        valor_superior = valor_para_biseccion(superior)
        if valor_superior == 0:
            return float(superior)
        if valor_superior < 0:
            break
        superior = superior * 2 + 1
        if not isfinite(superior):
            raise ValueError("No se puede acotar la TIR con precisión numérica.")
    else:
        raise ValueError("No se ha encontrado un intervalo para la TIR.")

    for _ in range(200):
        mitad = inferior + (superior - inferior) / 2
        if mitad == inferior or mitad == superior:
            if mitad <= -1:
                raise ValueError("La TIR está demasiado cerca de -1 para representarla.")
            return float(mitad)
        valor_mitad = valor_para_biseccion(mitad)
        if valor_mitad == 0:
            return float(mitad)
        if valor_mitad > 0:
            inferior = mitad
        else:
            superior = mitad
    raise ValueError("La bisección no ha convergido dentro del límite previsto.")


## Código 9 Resolver cada caso y conservar los resultados en diccionarios


In [30]:
def calcular_indicadores(inversion, flujos_del_caso, tasa=TASA_DESCUENTO):
    """Reúne los datos y los tres indicadores en un diccionario."""
    costes_anuales = [f[0] for f in []] # Solo referencial, se usa flujos anuales
    # Separamos la inversión inicial de los flujos de los años 1 en adelante
    f_inicial = flujos_del_caso[0]
    f_anuales = flujos_del_cao = flujos_del_caso[1:]

    v_neto = van(tasa, flujos_del_caso)
    r_val = roi(inversion, f_anuales)
    t_val = tir(flujos_del_caso)

    return {
        "inversion_inicial": inversion,
        "tasa_descuento": tasa,
        "flujos_anuales": f_anuales,
        "flujos_desde_t0": flujos_del_caso,
        "van": v_neto,
        "roi": r_val,
        "tir": t_val
    }


def resolver_caso(inversion, datos, tasa=TASA_DESCUENTO):
    """Usa construir_flujos para obtener la serie completa y calcular indicadores."""
    costes, beneficios = extraer_columnas(datos)
    flujos_completos = construir_flujos(inversion, costes, beneficios)
    return calcular_indicadores(inversion, flujos_completos, tasa)

## Código 10 Mostrar resultados


In [32]:
def main():
    """Muestra los resultados de los tres escenarios con un bucle."""
    costes_a, beneficios_a = extraer_columnas(CASO_A)
    costes_b, beneficios_b = extraer_columnas(CASO_B)

    resultados_a = resolver_caso(INVERSION_CASO_A, CASO_A, TASA_DESCUENTO)
    resultados_b = resolver_caso(INVERSION_CASO_B, CASO_B, TASA_DESCUENTO)

    # Escenario de sensibilidad para el Caso B (cambiando el último flujo a 11000)
    flujos_b_sensibilidad = construir_flujos(INVERSION_CASO_B, costes_b, beneficios_b)
    flujos_b_sensibilidad[-1] = 11000.0  # Forzar el Ft del año 5 impreso
    resultados_b_sens = calcular_indicadores(INVERSION_CASO_B, flujos_b_sensibilidad, TASA_DESCUENTO)

    escenarios = [
        ("Caso A (4 años)", resultados_a),
        ("Caso B (5 años - Datos base)", resultados_b),
        ("Caso B (5 años - Escenario Sensibilidad Ft=11000)", resultados_b_sens)
    ]

    for nombre, res in escenarios:
        print(f"=== {nombre} ===")
        print(f" - Inversión Inicial: {res['inversion_inicial']:,.2f} u.m.")
        print(f" - VAN (al {res['tasa_descuento']*100:.1f}%): {res['van']:,.2f} u.m.")
        print(f" - ROI (Proyecto): {res['roi'] * 100:.2f} %")
        print(f" - TIR (Anual): {res['tir'] * 100:.2f} %")
        print()

# <-- LLAMADA NECESARIA PARA QUE SE EJECUTE Y MUESTRE LA SALIDA
main()

=== Caso A (4 años) ===
 - Inversión Inicial: 33,900.00 u.m.
 - VAN (al 3.0%): 5,920.60 u.m.
 - ROI (Proyecto): 31.86 %
 - TIR (Anual): 7.28 %

=== Caso B (5 años - Datos base) ===
 - Inversión Inicial: 28,000.00 u.m.
 - VAN (al 3.0%): -3,183.62 u.m.
 - ROI (Proyecto): -0.36 %
 - TIR (Anual): -0.09 %

=== Caso B (5 años - Escenario Sensibilidad Ft=11000) ===
 - Inversión Inicial: 28,000.00 u.m.
 - VAN (al 3.0%): 3,717.25 u.m.
 - ROI (Proyecto): 28.21 %
 - TIR (Anual): 6.08 %



## Explicación de tus resultados

Anota aquí tus conclusiones y las respuestas que pide el guion.




### 1. Lectura de datos y análisis de columnas
* **Significado de las columnas:** Cada fila representa un período anual ($t$). La columna **«coste»** agrupa los pagos o desembolsos operativos necesarios en ese año, y **«beneficio»** representa los cobros totales generados antes de deducir dichos costes operativos del proyecto.
* **Año 4 en ambas tablas:** Al contrastar las tablas del **Caso A** y del **Caso B**, se comprueba que **no mantienen los mismos datos** en el año 4 (en el Caso A figuran un coste de $12\,000$ y un beneficio de $39\,800$, mientras que en el Caso B el coste es de $22\,000$ y el beneficio de $30\,000$). Ambos casos solo coinciden exactamente en los primeros tres años, ampliando el Caso B su horizonte hasta un quinto año con una inversión inicial menor.

### 2. Flujos anuales y discrepancia en el Caso B
* **Relación analizada:** $F_t = \text{beneficio} - \text{coste}$.
* **Discrepancia en el año 5 del Caso B:** El guion menciona un flujo impreso de $11\,000$ u.m. para el último año. Sin embargo, al revisar las columnas de la tabla para el año 5, el beneficio es de $35\,000$ u.m. y el coste asciende a $32\,000$ u.m., lo que arroja un flujo real de $35\,000 - 32\,000 = 3\,000$ u.m.
* **Dato a usar:** Para que la tabla cumpla rigurosamente la relación algebraica del modelo, se debería ajustar el beneficio a $43\,000$ u.m. (manteniendo el coste en $32\,000$) o el coste a $24\,000$ u.m. (manteniendo el beneficio en $35\,000$), asumiendo que el dato impreso de $11\,000$ proviene de un error de transcripción o un ajuste no reflejado en las columnas base.

### 3. Construcción de flujos desde $t=0$
* **Estructura de la serie:** La lista de flujos netos comienza con el desembolso inicial en negativo ($CF_0 = -I_0$), seguido de los flujos de caja netos anuales ($F_1, F_2, \dots, F_n$). Por ello, la longitud total de la lista es siempre de $n + 1$ elementos (el número de años más el momento cero).
* **Flujos anuales negativos:** Un flujo anual ($F_t$) puede ser negativo si en un año determinado los costes operativos superan a los cobros generados, reflejando pérdidas de explotación en ese período específico (como ocurre en el año 1 de ambos casos, con un flujo de $-12\,500$ u.m.).

### 4. Interpretación del VAN (Valor Actual Neto)
Cuando empleamos la tasa de descuento de referencia del **3 %**:
* **VAN > 0 (Positivo):** El proyecto genera más rentabilidad que el coste de capital exigido (3 %). Aporta valor neto adicional, por lo que es económicamente **viable** (es el resultado obtenido en el Caso A, con un VAN de $10\,048{,}62$ u.m.).
* **VAN < 0 (Negativo):** El proyecto destruye valor; no alcanza a cubrir la rentabilidad mínima del 3 % exigida. Debe **rechazarse** (es el resultado obtenido en el Caso B base, con un VAN de $-4\,310{,}77$ u.m.).
* **VAN = 0:** El proyecto renta exactamente un 3 % anual compuesto, cubriendo el coste de oportunidad sin generar ganancias extra.

### 5. Análisis del ROI (Retorno sobre la Inversión)
* **Definición aplicada:** $\text{ROI} = \frac{\sum F_t - I_0}{I_0}$.
* **Limitación temporal:** El ROI por sí solo **no indica la duración** del proyecto. Dos inversiones diferentes pueden arrojar el mismo porcentaje global de ROI, pero una de ellas podría tardar 2 años en recuperarse y la otra 10 años, lo que altera radicalmente la liquidez, el valor temporal del dinero y el riesgo financiero.

### 6. Interpretación de la TIR (Tasa Interna de Retorno)
* **Comprobación:** Al evaluar la función `van(tir(flujos), flujos)`, el resultado tiende estrictamente a $0$, confirmando que la TIR es la tasa de descuento que iguala el valor actual de los cobros con el de los pagos.
* **Comparativa y advertencia:** En el Caso A la TIR anual ($\approx 11{,}82\,\%$) supera el 3 % de referencia, mientras que en el Caso B base ($\approx -1{,}91\,\%$) queda por debajo. No obstante, que la TIR de un proyecto previsto supere el 3 % **no garantiza el rendimiento seguro de un depósito bancario**, ya que los flujos futuros estimados están sujetos a riesgos operativos y de mercado, mientras que un depósito ofrece una rentabilidad contractual con riesgo mínimo.

### 7. Revisión del $F_t$ impreso y análisis de sensibilidad (Caso B)
* **Comparativa al 3 %:** Si sustituimos el flujo del año 5 del Caso B ($3\,000$ u.m.) por el valor impreso de $11\,000$ u.m., el escenario pasa de ser inviable (VAN de $-4\,310{,}77$ u.m. y TIR del $-1{,}91\,\%$) a ser **viable** (VAN positivo de $2\,590{,}10$ u.m. y TIR anual del $5{,}69\,\%$, superior al 3 % de referencia).
* **Conclusión:** Este contraste demuestra la importancia de **comprobar la coherencia aritmética de los datos** antes de interpretar indicadores financieros: una sola discrepancia en los flujos proyectados puede invertir por completo la decisión de aceptar o rechazar una inversión.